In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from sklearn.preprocessing import StandardScaler

# Load combined gene-expression dataset
df = pd.read_csv("ALL_AML_combined.csv")

# Rows = genes, columns = patient IDs
expression = df.set_index("Gene Accession Number")
expression = expression[[str(i) for i in range(1, 73)]]

# Original training labels
# Patients 1–27: ALL (+1)
# Patients 28–38: AML (-1)
all_train = [str(i) for i in range(1, 28)]
aml_train = [str(i) for i in range(28, 39)]

# Welch's t-test across all genes
t_stat, p_values = ttest_ind(
    expression[all_train],
    expression[aml_train],
    axis=1,
    equal_var=False,
    nan_policy="omit"
)

# Select 8 genes with smallest p-values
p_values = pd.Series(p_values, index=expression.index)
top8_genes = p_values.dropna().nsmallest(8).index.tolist()

print("Selected genes:", top8_genes)

# Alternatively, manually specify the 8 gene accession numbers:
# top8_genes = ["gene1", "gene2", ..., "gene8"]

# Transpose: rows = patients, columns = 8 selected genes
X = expression.loc[top8_genes].T

# Normalize using training patients only
scaler = StandardScaler()
scaler.fit(X.loc[[str(i) for i in range(1, 39)]])

X_scaled = pd.DataFrame(
    scaler.transform(X),
    index=X.index,
    columns=X.columns
)

# ALL = +1, AML = -1 (all 72 patients)
all_ids = (
    set(range(1, 28))
    | set(range(39, 50))
    | {55, 56, 59}
    | set(range(67, 73))
)

labels = {
    i: (1 if i in all_ids else -1)
    for i in range(1, 73)
}

# Choose two samples, one ALL and one AML
sample_ids = [1, 28]

# Include labels, as in the original mock data
small_data = [
    X_scaled.loc[str(i)].tolist() + [labels[i]]
    for i in sample_ids
]

# Remove labels for inner-product calculations
train_data = [row[:-1] for row in small_data]

print("small_data:", small_data)
print("train_data:", train_data)

train_data = [small_data[0][:-1], small_data[1][:-1]]

In [ ]:
# from qiskit.circuit.library import zz_feature_map
# fm = zz_feature_map(feature_dimension=np.shape(train_data)[1], entanglement='linear', reps=1)

from qiskit.circuit.library import z_feature_map

fm = z_feature_map(feature_dimension=np.shape(train_data)[1])


unitary1 = fm.assign_parameters(train_data[0])
unitary2 = fm.assign_parameters(train_data[1])

In [ ]:
from qiskit.circuit.library import unitary_overlap


overlap_circ = unitary_overlap(unitary1, unitary2)
overlap_circ.measure_all()

print("circuit depth = ", overlap_circ.decompose().depth())
overlap_circ.decompose().draw("mpl", scale=0.6, style="iqp")